# rtp-audit: one experiment cycle on Colab (G4 / RTX PRO 6000 96 GB, or larger)

**Open this notebook straight from GitHub:** [Open in Colab](https://colab.research.google.com/github/GuangLun2000/RTP-Audit/blob/main/colab/run_cycle.ipynb). Colab then loads the newest pushed version every time; there is no copy to keep on Drive (if you do save a copy, cell 2 warns when it is outdated).

A **cycle** is one short, budgeted GPU session that answers one question (`configs/cycles/*.yaml`): at most 12 hours, usually far less. Set `CYCLE` in cell 1, then run the cells top to bottom.

- **Code** comes from GitHub: cell 2 clones the repository into `/content/rtp-audit` (local disk) and updates it to the newest push of the pinned branch. For a confirmation cycle set `PIN` to a commit hash.
- **Results** go to `RTP_WORK_DIR`. With `USE_DRIVE = True` (cell 1) that is `MyDrive/rtp` on Google Drive, so finished shards survive a disconnect.
- **Before running:** add `HF_TOKEN` in Colab *Secrets*.

How a session goes:
1. Cells 1-5 prepare the environment, run the smoke test and build the items (minutes).
2. Cell 6 prices the cycle: every job with its predicted time. A cycle that does not fit its budget is refused before any GPU time is spent.
3. Cell 7 runs the jobs in priority order. A cycle never mixes results: if a stage of an earlier cycle was built by other code, items or settings, cell 6 stops and says so; set `AFTER_INSTRUMENT_CHANGE = True` in cell 1 for that session to rebuild the items and re-measure those stages. After every job the scorecard is rewritten, so an interrupted session still leaves usable results; after a disconnect re-run cells 1-3 and cell 7 (finished shards and stages are reused; jobs that no longer fit the budget are deferred to the next session).
4. Cell 8 shows the scorecard; cell 9 writes the *feedback bundle*. Give that file to the coding agent ([docs/AGENT_LOOP.md](../docs/AGENT_LOOP.md)).

The last cell (10) releases the GPU when everything above has run: it waits 10 seconds, lets Drive finish writing and ends the session. Set `RELEASE_GPU_WHEN_DONE = False` in cell 1 to keep the session open. If a cell stops with an error, "Run all" stops there and the session stays open.


In [ ]:
# Cell 1: settings, then results storage and environment variables
NOTEBOOK_REV = "84c0d98361"  # identifies this notebook's code cells; cell 2 compares it with the repo's copy
REPO_URL = "https://github.com/GuangLun2000/RTP-Audit.git"
PIN = "main"  # a branch (updated to its newest push) or a commit hash (exactly that code; use it for confirm cycles)
USE_DRIVE = True                       # results on Drive survive disconnects; False = /content/rtp_runs (lost on reset)
CYCLE = "configs/cycles/c0.yaml"       # the cycle to run (its file names the run config, the jobs and the budget)
CONFIG = "configs/e1.yaml"             # the run config of that cycle (doctor, data, feedback)
AFTER_INSTRUMENT_CHANGE = False        # True for the first session after the items, the readout or the config changed:
                                       # rebuilds the items and re-measures the stages they made stale (develop phase only)
RELEASE_GPU_WHEN_DONE = True           # last cell: wait 10 s, sync Drive, end the session so the GPU is released

import os
from google.colab import userdata

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ["RTP_WORK_DIR"] = "/content/drive/MyDrive/rtp"
else:
    os.environ["RTP_WORK_DIR"] = "/content/rtp_runs"
os.environ["HF_HOME"] = "/content/hf"          # model cache on local disk, not Drive
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:                        # missing secret: gated Llama repos will fail, others work
    print("⚠️ HF_TOKEN secret not available:", type(exc).__name__)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["RTP_PROGRESS"] = "log"              # readable progress lines instead of carriage-return bars
os.makedirs(os.environ["RTP_WORK_DIR"], exist_ok=True)
print("results in", os.environ["RTP_WORK_DIR"])


In [ ]:
# Cell 2: get the code from GitHub and install it (torch is never reinstalled)
# Stops with ❌ and a reason at the first failing step; ends with "✅ code ready" when everything worked.
import importlib, json, os, re, subprocess, sys

REPO_DIR = "/content/rtp-audit"
OPEN_LINK = "https://colab.research.google.com/github/GuangLun2000/RTP-Audit/blob/main/colab/run_cycle.ipynb"


def sh(*cmd, cwd=None, check=True):
    try:
        r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    except FileNotFoundError:
        if check:
            raise RuntimeError(f"❌ program not found: {cmd[0]}")
        print("$", " ".join(cmd), "\n(not available)")
        return None
    out = (r.stdout + r.stderr).strip()
    print("$", " ".join(cmd) + ("" if not out else "\n" + out[-3000:]))
    if check and r.returncode:
        raise RuntimeError(f"❌ step failed (exit {r.returncode}): {' '.join(cmd)}")
    return r


sh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv", check=False)
if not str(REPO_URL).startswith("https://"):
    raise RuntimeError(f"❌ REPO_URL is not set: this is an old copy of the notebook. Open it from GitHub: {OPEN_LINK}")
if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh("git", "clone", "-q", REPO_URL, REPO_DIR)
sh("git", "fetch", "-q", "--all", "--tags", cwd=REPO_DIR)
sh("git", "checkout", "-q", PIN, cwd=REPO_DIR)
if sh("git", "symbolic-ref", "-q", "HEAD", cwd=REPO_DIR, check=False).returncode == 0:  # a branch: newest push
    sh("git", "pull", "-q", "--ff-only", cwd=REPO_DIR)
os.chdir(REPO_DIR)

import torch

torch_before = torch.__version__
pip = sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]", cwd=REPO_DIR)
if "dependency resolver" in (pip.stdout + pip.stderr):
    print("ℹ️ pip's resolver note above is expected: the project pins transformers<5, so pip downgrades Colab's "
          "transformers 5.x, and preinstalled packages that rtp never imports (e.g. sentence-transformers) now report "
          "a conflict. The install itself succeeded (exit 0).")
ver = sh(sys.executable, "-c", "import torch, transformers, rtp; print(torch.__version__, transformers.__version__)",
         cwd=REPO_DIR).stdout.split()
if ver[0] != torch_before:
    raise RuntimeError(f"❌ torch changed from {torch_before} to {ver[0]}; restart the runtime (never reinstall torch)")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
repo_nb = json.load(open(f"{REPO_DIR}/colab/run_cycle.ipynb", encoding="utf-8"))
m = re.search(r'(?m)^NOTEBOOK_REV = "([^"]+)"', "".join("".join(c["source"]) for c in repo_nb["cells"]))
if m and m.group(1) != NOTEBOOK_REV:
    print(f"⚠️ this notebook (rev {NOTEBOOK_REV}) differs from the repo's (rev {m.group(1)}). Reopen it from GitHub: "
          f"{OPEN_LINK}")
head = sh("git", "log", "-1", "--format=%h %s", cwd=REPO_DIR).stdout.strip()
print(f"✅ code ready: {head} | torch {ver[0]} (unchanged) | transformers {ver[1]}")


In [ ]:
# Cell 3: preflight. GPU, bf16, library pins, HF token, disks, Drive, data, Hub access to the config's models
import importlib

try:
    import rtp.notebook as nb
except ModuleNotFoundError as exc:
    raise RuntimeError("❌ the rtp package is not importable: run cell 2 first; it must end with '✅ code ready'") from exc
importlib.reload(nb)          # pick up helper changes after a `git pull` without restarting the runtime
nb.ensure_peft()             # Colab ships a torchao that peft refuses: LoRA adapters would fail later
nb.run("doctor", "--config", CONFIG)
RUN = f"{os.environ['RTP_WORK_DIR']}/runs/{nb.config_name(CONFIG)}"
CYCLE_DIR = f"{RUN}/cycles/{nb.cycle_name(CYCLE)}"


In [ ]:
# Cell 4: smoke test and the test suite (a few minutes)
nb.run("smoke")
nb.pytest(CONFIG)             # RTP_REQUIRE_INTEGRATION=1: on Colab the smoke model must load, skips count as failures


In [ ]:
# Cell 5: generate the items (only once; reused afterwards)
nb.run("build-data", "--config", CONFIG, *(["--force"] if AFTER_INSTRUMENT_CHANGE else []), strict=True)
nb.show(f"{RUN}/data/data_report.md", max_chars=5000)


In [ ]:
# Cell 6: price the cycle. Every job with its state and predicted time; a cycle over its budget is refused here.
REDO = ["--redo-stale"] if AFTER_INSTRUMENT_CHANGE else []
nb.run("cycle", "plan", "--cycle", CYCLE, *REDO, strict=True)


In [ ]:
# Cell 7: run the cycle (resumable: re-run this cell after an interruption; batch sizes are set for this GPU)
nb.run("cycle", "run", "--cycle", CYCLE, *REDO)
nb.run("status", "--config", CONFIG)


In [ ]:
# Cell 8: the scorecard. It starts with the FINDINGS (what the numbers mean and what to do next),
# then the tables: checks per model, the ruler, the brands on it, the stage pairs, the budget ledger.
nb.show(f"{CYCLE_DIR}/scorecard.md", max_chars=120000)


In [ ]:
# Cell 9: feedback bundle for the coding agent (run it after every cycle, and whenever something failed).
# Give the agent the downloaded FILE feedback_<UTC>.md (also on Drive: runs/<config>/feedback/),
# not this notebook's printed output: the file holds the stage records and diagnostics as well.
bundle = nb.feedback(CONFIG)
if bundle:
    nb.download(bundle)


In [ ]:
# Cell 10: release the GPU. Waits 10 seconds (the bundle download finishes), lets Drive finish writing,
# then ends the session. Kept open when the results are not on Drive (USE_DRIVE = False).
if RELEASE_GPU_WHEN_DONE:
    nb.release_gpu(wait_seconds=10)
